# Two-stage sleep-stage and awakening forecasting

This notebook trains a stage classifier first, then an awakening forecaster. For each completed 30-second epoch, the classifier produces probabilities for Wake, N1, N2, N3, and REM. Those five probabilities are concatenated with the 11 normalized EEG features, so the causal forecaster receives a `(batch, 10, 16)` sequence and predicts 13 awakening-time classes.

The forecaster uses a left-padded causal 1D convolution and two causal Transformer encoder layers. At each position, attention can use that position and earlier positions only. The stage classifier is trained with class-weighted cross-entropy; the forecaster is trained with multiclass cross-entropy. Training examples use subject-grouped out-of-fold stage predictions to avoid unrealistically clean inputs. Validation forecast log loss selects the forecaster checkpoint. Test subjects are evaluated only after selection.

First run `Sleep_EDF_Wake_Up_Prediction_Colab.ipynb` and download its schema-v2 archive. The existing schema-v1 archive has no detailed stage labels. Preserve the original subject splits. To use the new archive in the hosted Admin app, open **Data**, upload it as a version of the original dataset, then select that five-stage version in **Train a model**. This notebook can also train/evaluate locally in Colab.


In [ ]:
!git clone --branch sleepops --depth 1 https://github.com/adam12bT/deepecg.git /content/deepecg
%cd /content/deepecg
%pip -q install -e '.[train]'


## Upload the prepared schema-v2 archive

Upload the `.npz` exported by the preprocessing notebook. The validator prints keys, array shapes, split sizes, label counts, contiguous histories, and verifies 13-class targets against the annotated sustained-Wake onset.


In [ ]:
from google.colab import files
from pathlib import Path

uploaded = files.upload()
archive = Path(next(name for name in uploaded if name.endswith('.npz')))
print(f'Using {archive.name}')


In [ ]:
import json
import subprocess
import sys

runtime = Path('/content/sleepops_runtime')
runtime.mkdir(parents=True, exist_ok=True)
lock = Path('/content/stage_dataset.json')
subprocess.run([sys.executable, '-m', 'sleepops', 'validate', '--archive', str(archive), '--write', str(lock)], check=True)
manifest = json.loads(lock.read_text())
print('Five-class counts by split (Wake, N1, N2, N3, REM):')
print(json.dumps({split: stats.get('stage_counts') for split, stats in manifest['splits'].items()}, indent=2))


## Architecture, tensor shapes, and causality

The raw model input is `x: (B, 10, 11)`: batch, ten chronological 30-second epochs, and eleven normalized EEG features. The first-stage MLP classifies each epoch independently and returns `stage_logits: (B, 10, 5)`. Softmax turns those into five stage probabilities per epoch. The pipeline concatenates probabilities to the original EEG features, giving `z: (B, 10, 16)`.

The forecaster transposes `z` to `(B, 16, 10)` for a one-dimensional convolution. A kernel of width three is padded with two zeros on the left and none on the right, so all ten output positions are preserved and position *t* never sees an EEG epoch after *t*. The convolution maps each position to 64 features; positional embeddings keep the epoch order explicit.

Two Transformer encoder layers use four attention heads (16 features per head), a 128-unit feed-forward block, residual connections, and layer normalization. Each query uses a lower-triangular causal attention mask: it can attend to its own position and earlier positions, never later ones. The final position is `(B, 64)` and its linear forecast head returns 13 logits: twelve five-minute awakening bins and one no-awakening-before-60-minutes class. Softmax is applied only to report forecast probabilities.

The sleep/wake output is derived at each epoch as `P(Asleep) = 1 - P(Wake)` from the five-class classifier; the v2 cascade therefore trains with weighted five-class cross-entropy for stages and 13-class cross-entropy for awakening time, rather than a separate binary BCE head. Validation stage cross-entropy selects the classifier training duration; validation forecast cross-entropy selects the final forecasting checkpoint. A causality check perturbs every possible later suffix and asserts that earlier stage predictions stay unchanged. At deployment, every input is limited to the ten completed EEG epochs available at that prediction time.

## Train with validation-only model selection

Normalization means and scales are fit on training epochs only. The sleep-stage classifier is selected by validation stage cross-entropy. Its out-of-fold probabilities are generated by holding out whole training subjects, then joined to the EEG features. The awakening network is selected by validation forecast cross-entropy. This avoids using future EEG at inference and avoids using the test set for tuning.


In [ ]:
import torch

command = [sys.executable, '-m', 'sleepops', '--runtime', str(runtime), 'train',
           '--archive', str(archive), '--lock', str(lock), '--epochs', '20',
           '--device', 'cuda' if torch.cuda.is_available() else 'cpu']
subprocess.run(command, check=True)
runs = sorted((runtime / 'runs').iterdir(), key=lambda path: path.stat().st_mtime, reverse=True)
run_dir = runs[0]
run_id = run_dir.name
print(f'Selected run {run_id}')


## Final evaluation on held-out subjects

This step is deliberately after model selection. It reports five-class stage metrics and the derived sleep/wake comparison to logistic regression, plus awakening forecast log loss and within-hour Brier score against the training-class distribution. It also saves a probability plot from an unseen test recording.


In [ ]:
subprocess.run([sys.executable, '-m', 'sleepops', '--runtime', str(runtime), 'evaluate',
                '--archive', str(archive), '--lock', str(lock), '--run', run_id,
                '--device', 'cuda' if torch.cuda.is_available() else 'cpu'], check=True)
for name in ('validation_metrics.json', 'test_metrics.json'):
    report = json.loads((run_dir / name).read_text())
    print(name)
    print(json.dumps({'stages': report.get('stages'), 'sleep_wake': report['state'],
                      'forecast': report['forecast'], 'subjects': report['subjects']}, indent=2))


## Download the checkpoint and reports

The checkpoint bundles the stage classifier, causal awakening forecaster, feature order, and training-only normalization statistics used at inference. The report and plot record the actual results from this Colab run. No stage-aware test results exist until you run this notebook with the stage-labeled archive.


In [ ]:
from zipfile import ZipFile, ZIP_DEFLATED

bundle = Path('/content/sleepops_stage_run.zip')
with ZipFile(bundle, 'w', ZIP_DEFLATED) as archive_zip:
    for name in ('best_checkpoint.pt', 'training_log.json', 'validation_metrics.json',
                 'test_metrics.json', 'heldout_forecast.png'):
        path = run_dir / name
        if path.is_file():
            archive_zip.write(path, arcname=name)
files.download(str(bundle))
